# Huấn Luyện DF-FusionModel CAN — Giai đoạn 2 & 3 (Curriculum Fine-tuning)

Sổ tay thực nghiệm thực hiện huấn luyện tiếp nối (Curriculum Fine-tuning) cho mô hình DF-FusionModel CAN:
1. **Giai đoạn 2 (Stage 2 - rPPG Fine-tuning):** Mở khóa (unfreeze) các tầng tích chập và Temporal Transformer của nhánh DeepPhysCAN, áp dụng tốc độ học nhỏ LR = 5e-6 để tinh chỉnh trích xuất nhịp mạch sinh học trên miền video nén FaceForensics++ C23. Nhánh Swin Transformer tiếp tục được đóng băng để bảo toàn biểu diễn đặc trưng không gian.
2. **Giai đoạn 3 (Stage 3 - End-to-End Fine-tuning):** Mở khóa toàn bộ mạng bao gồm cả Swin Transformer với LR = 1e-6 nhằm tối ưu hóa liên kết toàn hệ thống.
3. **Theo dõi Checkpoint:** Lưu trữ độc lập checkpoint tối ưu cho từng giai đoạn (`fusion_model_stage2_best.pth`, `fusion_model_stage3_best.pth`).

---
### Cấu hình tập dữ liệu đầu vào:
- **Spatial Cache:** `ffpp_c23_cache` (bộ đệm khuôn mặt căn chỉnh 224x224).
- **rPPG Cache:** `ffpp_c23_rppg_cache` (ten-xơ vi sai chuẩn hóa NFD 96x96).
- **Stage 1 Checkpoint:** `fusion_model_best.pth` (trọng số tiền huấn luyện đầu hợp nhất Cross-Attention).

In [ ]:
!pip install -q timm


In [ ]:
import os, glob, time, random, copy, json, math, sys
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# 
# 1. TỰ ĐỘNG PHÁT HIỆN ĐƯỜNG DẪN TRÊN KAGGLE (ROBUST AUTO-DISCOVERY)
# 
def find_dir(patterns, default_fallback):
    for pat in patterns:
        for p in glob.glob(pat, recursive=True):
            if os.path.isdir(p):
                return p
    return default_fallback

def find_file(patterns, default_fallback):
    for pat in patterns:
        for p in glob.glob(pat, recursive=True):
            if os.path.isfile(p):
                return p
    return default_fallback

# A. Thư mục Cache dữ liệu
CACHE_DIR_SPATIAL = find_dir([
    "/kaggle/input/**/ffpp_c23_cache",
    "/kaggle/input/datasets/thinhphan272/ffpp-c23-face-cache/ffpp_c23_cache",
    "/kaggle/input/ffpp-c23-face-cache/ffpp_c23_cache",
], "/kaggle/input/datasets/thinhphan272/ffpp-c23-face-cache/ffpp_c23_cache")

CACHE_DIR_RPPG = find_dir([
    "/kaggle/input/**/ffpp_c23_rppg_cache",
    "/kaggle/input/datasets/thinhphan272/ffpp-c23-rppg-cache/ffpp_c23_rppg_cache",
    "/kaggle/input/ffpp-c23-rppg-cache/ffpp_c23_rppg_cache",
], "/kaggle/input/datasets/thinhphan272/ffpp-c23-rppg-cache/ffpp_c23_rppg_cache")

# B. Checkpoint Stage 1 (để tiếp tục train Stage 2 & 3)
FUSION_STAGE1_CKPT = find_file([
    "/kaggle/input/**/fusion_model_best.pth",
    "/kaggle/input/datasets/thinhphan1506/trained-dffusionmodel-can-pth/fusion_model/fusion_model_best.pth",
    "/kaggle/input/trained-dffusionmodel-can-pth/fusion_model/fusion_model_best.pth",
], "")

# C. Checkpoint Backbone riêng lẻ (Spatial và rPPG)
SPATIAL_CKPT_PATH = find_file([
    "/kaggle/input/**/spatial_transformer_best.pth",
    "/kaggle/input/datasets/thinhphan1506/dffusion-trained-pth/spatial_transformer/spatial_transformer_best.pth",
    "/kaggle/input/dffusion-trained-pth/spatial_transformer/spatial_transformer_best.pth",
], "/kaggle/input/datasets/thinhphan1506/dffusion-trained-pth/spatial_transformer/spatial_transformer_best.pth")

RPPG_CKPT_PATH = find_file([
    "/kaggle/input/**/rppg_can_best.pth",
    "/kaggle/input/datasets/thinhphan272/trained-dffusionmodel-rppg-pth/rppg_can_branch/rppg_can_best.pth",
    "/kaggle/input/trained-dffusionmodel-rppg-pth/rppg_can_branch/rppg_can_best.pth",
], "/kaggle/input/datasets/thinhphan272/trained-dffusionmodel-rppg-pth/rppg_can_branch/rppg_can_best.pth")

print("="*65)
print("=== THÔNG TIN ĐƯỜNG DẪN DỮ LIỆU & CHECKPOINT ===")
print("="*65)
print(f"CACHE_DIR_SPATIAL : {CACHE_DIR_SPATIAL} -> {'OK' if os.path.isdir(CACHE_DIR_SPATIAL) else 'KHÔNG TÌM THẤY'}")
print(f"CACHE_DIR_RPPG    : {CACHE_DIR_RPPG} -> {'OK' if os.path.isdir(CACHE_DIR_RPPG) else 'KHÔNG TÌM THẤY'}")
has_stage1 = bool(FUSION_STAGE1_CKPT and os.path.isfile(FUSION_STAGE1_CKPT))
print(f"FUSION_STAGE1_CKPT: {FUSION_STAGE1_CKPT if has_stage1 else 'Không có (Sẽ train từ Stage 1)'}")
print(f"SPATIAL_CKPT_PATH : {SPATIAL_CKPT_PATH} -> {'OK' if os.path.isfile(SPATIAL_CKPT_PATH) else 'KHÔNG TÌM THẤY'}")
print(f"RPPG_CKPT_PATH    : {RPPG_CKPT_PATH} -> {'OK' if os.path.isfile(RPPG_CKPT_PATH) else 'KHÔNG TÌM THẤY'}")
print("="*65)

# Tự động quyết định Stage bắt đầu:
# Nếu có file Stage 1 checkpoint -> Bắt đầu từ Stage 2 (tiết kiệm thời gian)
# Nếu không có -> Bắt đầu từ Stage 1
START_FROM_STAGE = 2 if has_stage1 else 1
print(f"==> KẾT QUẢ: Bắt đầu huấn luyện từ STAGE {START_FROM_STAGE}!")

# 
# 2. THAM SỐ HUẤN LUYỆN
# 
SPATIAL_BACKBONE_NAME = "swin_base_patch4_window7_224"
SPATIAL_IMG_SIZE      = 224
N_SPATIAL_FRAMES_TRAIN = 6
N_SPATIAL_FRAMES_EVAL  = 12

CAN_IMG_SIZE        = 96
RPPG_CLIP_LEN_TRAIN = 48
RPPG_CLIP_LEN_EVAL  = 96
DIFF_EPS            = 0.01
MIN_FRAMES_RPPG     = 50

FUSION_PROJ_DIM     = 384
FUSION_HEADS        = 4
DROPOUT             = 0.25

BATCH_SIZE          = 4
GRAD_ACCUM          = 4        # effective batch = 16

# Số epoch cho từng Stage
EPOCHS_STAGE1       = 10       # Freeze cả 2 backbone, train Cross-Attention
EPOCHS_STAGE2       = 8        # Finetune rPPG backbone + Fusion head
EPOCHS_STAGE3       = 6        # Full Finetune toàn bộ mạng

LR_FUSION           = 4e-4
LR_FINETUNE_RPPG    = 8e-6     # Tinh chỉnh rPPG backbone
LR_FINETUNE_ALL_FUS = 2e-5     # Tinh chỉnh Fusion head trong Stage 3
LR_FINETUNE_ALL_SP  = 2e-6     # Tinh chỉnh Swin spatial trong Stage 3
WEIGHT_DECAY        = 1e-2

MIXUP_ALPHA         = 0.2
LABEL_SMOOTH_EPS    = 0.05
EMA_DECAY           = 0.999
PATIENCE            = 6
SEED                = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR = "/kaggle/working/fusion_model"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print(f"Device: {DEVICE} | Output Dir: {OUT_DIR}")


In [ ]:
def load_manifest(cache_dir):
    p = os.path.join(cache_dir, "manifest_processed.csv")
    if not os.path.exists(p):
        candidates = glob.glob(os.path.join(cache_dir, "**", "manifest_processed.csv"), recursive=True)
        if candidates: p = candidates[0]
        else: raise FileNotFoundError(f"Không tìm thấy manifest_processed.csv trong {cache_dir}")
    m = pd.read_csv(p)
    # Thiết lập đường dẫn out_dir
    if "out_dir" in m.columns:
        first_valid = m["out_dir"].dropna().iloc[0] if len(m) > 0 else ""
        if "ffpp_c23_cache" in first_valid:
            prefix = first_valid.split("ffpp_c23_cache")[0] + "ffpp_c23_cache"
            m["out_dir"] = m["out_dir"].astype(str).str.replace(prefix, cache_dir, regex=False)
        elif "ffpp_c23_rppg_cache" in first_valid:
            prefix = first_valid.split("ffpp_c23_rppg_cache")[0] + "ffpp_c23_rppg_cache"
            m["out_dir"] = m["out_dir"].astype(str).str.replace(prefix, cache_dir, regex=False)
    return m

manifest_spatial = load_manifest(CACHE_DIR_SPATIAL)
manifest_spatial = manifest_spatial[manifest_spatial["status"].isin(["ok", "skip_exists"])]

manifest_rppg = load_manifest(CACHE_DIR_RPPG)
manifest_rppg = manifest_rppg[
    manifest_rppg["status"].isin(["ok", "skip_exists", "partial"]) & (manifest_rppg["n_frames"] >= MIN_FRAMES_RPPG)
]

# Kiểm tra trùng lặp
manifest_spatial = manifest_spatial.drop_duplicates(subset=["video_id", "method"]).reset_index(drop=True)
manifest_rppg = manifest_rppg.drop_duplicates(subset=["video_id", "method"]).reset_index(drop=True)

# Merge theo (video_id, method)
merged = manifest_spatial[["video_id", "split", "label", "method", "out_dir"]].rename(
    columns={"out_dir": "spatial_dir"}
).merge(
    manifest_rppg[["video_id", "method", "out_dir", "label"]].rename(
        columns={"out_dir": "rppg_dir", "label": "label_rppg"}),
    on=["video_id", "method"], how="inner"
)
assert (merged["label"] != merged["label_rppg"]).sum() == 0, "Nhãn không khớp giữa 2 manifest!"
merged = merged.drop(columns=["label_rppg"]).drop_duplicates(subset=["video_id", "method"]).reset_index(drop=True)

print(f"Tổng số video sau khi merge: {len(merged)}")
print(merged.groupby(["split", "label"])["video_id"].count().unstack(fill_value=0)
      .rename(columns={0: "real", 1: "fake"}))

train_df = merged[merged.split == "train"].reset_index(drop=True)
val_df   = merged[merged.split == "val"].reset_index(drop=True)
test_df  = merged[merged.split == "test"].reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_spatial_frame(path, augment=False):
    img = cv2.imread(path)
    if img is None: return torch.zeros(3, SPATIAL_IMG_SIZE, SPATIAL_IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (SPATIAL_IMG_SIZE, SPATIAL_IMG_SIZE))
    if augment and random.random() < 0.5:
        img = img[:, ::-1, :].copy()
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(img.transpose(2, 0, 1)).float()

def load_rppg_frame(path):
    img = cv2.imread(path)
    if img is None: return np.zeros((CAN_IMG_SIZE, CAN_IMG_SIZE, 3), dtype=np.float32)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (CAN_IMG_SIZE, CAN_IMG_SIZE)).astype(np.float32)

def build_rppg_clip(paths, clip_len, train=True):
    n = len(paths)
    need = clip_len + 1
    start = random.randint(0, max(n - need, 0)) if (train and n >= need) else max(0, (n - need) // 2)
    apps, mots, prev = [], [], None
    for i in range(min(need, n)):
        fi = min(start + i, n - 1)
        img = load_rppg_frame(paths[fi])
        if prev is not None:
            c, p = img / 255.0, prev / 255.0
            apps.append(c)
            mots.append((c - p) / (c + p + DIFF_EPS))
        prev = img
    while len(apps) < clip_len:
        apps.append(apps[-1] if apps else np.zeros((CAN_IMG_SIZE, CAN_IMG_SIZE, 3), np.float32))
        mots.append(mots[-1] if mots else np.zeros((CAN_IMG_SIZE, CAN_IMG_SIZE, 3), np.float32))
    app_t = torch.from_numpy(np.stack(apps[:clip_len])).permute(0, 3, 1, 2).float()
    mot_t = torch.from_numpy(np.stack(mots[:clip_len])).permute(0, 3, 1, 2).float()
    return app_t, mot_t

class FusionDataset(Dataset):
    def __init__(self, df, train=True):
        self.rows = df.reset_index(drop=True)
        self.train = train
        self.n_sp  = N_SPATIAL_FRAMES_TRAIN if train else N_SPATIAL_FRAMES_EVAL
        self.rppg_len = RPPG_CLIP_LEN_TRAIN if train else RPPG_CLIP_LEN_EVAL

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        sp_files = sorted(glob.glob(os.path.join(row.spatial_dir, "*.jpg")))
        if self.train:
            chosen = random.sample(sp_files, min(self.n_sp, len(sp_files)))
        else:
            step = max(len(sp_files) // self.n_sp, 1)
            chosen = sp_files[::step][:self.n_sp]
        while len(chosen) < self.n_sp:
            chosen.append(chosen[-1] if chosen else (sp_files[0] if sp_files else ""))
        sp_frames = torch.stack([load_spatial_frame(p, augment=self.train) for p in chosen])

        rppg_files = sorted(glob.glob(os.path.join(row.rppg_dir, "*.jpg")))
        app, mot = build_rppg_clip(rppg_files, self.rppg_len, train=self.train)

        label = torch.tensor(row.label, dtype=torch.float32)
        if self.train:
            return sp_frames, app, mot, label
        return sp_frames, app, mot, label, row.video_id

train_ds = FusionDataset(train_df, train=True)
val_ds   = FusionDataset(val_df,   train=False)
test_ds  = FusionDataset(test_df,  train=False)

# Weighted Random Sampler
class_counts = train_df["label"].value_counts().to_dict()
METHOD_WEIGHT = {"NeuralTextures": 2.0, "Face2Face": 1.3, "FaceSwap": 1.3, "Deepfakes": 1.0}

def get_weight(row):
    if row["label"] == 0: return 1.0 / max(class_counts.get(0, 1), 1)
    return (1.0 / max(class_counts.get(1, 1), 1)) * METHOD_WEIGHT.get(row.get("method", ""), 1.0)

weights = train_df.apply(get_weight, axis=1).values
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
print(f"DataLoaders sẵn sàng: Train={len(train_ds)} | Val={len(val_ds)} | Test={len(test_ds)}")


In [ ]:
import timm

# 1. Spatial Backbone (Swin Transformer) 
class SpatialTransformerDetector(nn.Module):
    def __init__(self, backbone_name=SPATIAL_BACKBONE_NAME, pretrained=False, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0,
                                           drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim), nn.Linear(feat_dim, 256), nn.GELU(), nn.Dropout(0.3), nn.Linear(256, 1))
    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)

# 2. rPPG Backbone (DeepPhysCAN_v2) 
class SqueezeExcitation(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                 nn.Linear(ch, ch // reduction), nn.ReLU(),
                                 nn.Linear(ch // reduction, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1).unsqueeze(-1)

class ResConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_se=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch))
        self.se = SqueezeExcitation(out_ch) if use_se else nn.Identity()
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.se(self.conv(x)) + self.shortcut(x))

class AttentionMask(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_ch, 1, 1), nn.Sigmoid())
    def forward(self, x):
        return self.conv(x)

class TemporalTransformer(nn.Module):
    def __init__(self, dim, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.pos_drop  = nn.Dropout(dropout)
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=dim, nhead=n_heads, dim_feedforward=dim * 2,
            dropout=dropout, activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm    = nn.LayerNorm(dim)
    def forward(self, x):
        B = x.shape[0]
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        return self.norm(self.encoder(self.pos_drop(x))[:, 0])

class DeepPhysCAN_v2(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.app_block1 = ResConvBlock(3, 48);   self.att_mask1 = AttentionMask(48)
        self.app_block2 = ResConvBlock(48, 96);  self.att_mask2 = AttentionMask(96)
        self.app_block3 = ResConvBlock(96, 192, use_se=True);  self.att_mask3 = AttentionMask(192)
        self.app_block4 = ResConvBlock(192, 256, use_se=True); self.att_mask4 = AttentionMask(256)
        self.mot_block1 = ResConvBlock(3, 48)
        self.mot_block2 = ResConvBlock(48, 96)
        self.mot_block3 = ResConvBlock(96, 192, use_se=True)
        self.mot_block4 = ResConvBlock(192, 256, use_se=True)
        self.pool = nn.AvgPool2d(2, 2)
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.frame_feat = nn.Sequential(nn.Linear(256, 256), nn.GELU(), nn.Dropout(dropout))
        self.temporal   = TemporalTransformer(dim=256, n_heads=4, n_layers=2, dropout=dropout)
        self.classifier = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.embed_dim  = 256

    def _forward_pair(self, app, mot):
        a1 = self.app_block1(app); m1 = self.att_mask1(a1); a1 = self.pool(a1)
        a2 = self.app_block2(a1);  m2 = self.att_mask2(a2); a2 = self.pool(a2)
        a3 = self.app_block3(a2);  m3 = self.att_mask3(a3); a3 = self.pool(a3)
        a4 = self.app_block4(a3);  m4 = self.att_mask4(a4)
        v1 = self.mot_block1(mot) * m1; v1 = self.pool(v1)
        v2 = self.mot_block2(v1) * m2;  v2 = self.pool(v2)
        v3 = self.mot_block3(v2) * m3;  v3 = self.pool(v3)
        v4 = self.mot_block4(v3) * m4
        return self.frame_feat(self.gap(v4).flatten(1))

    def forward_features(self, app_clip, mot_clip):
        B, T = app_clip.shape[:2]
        app_flat = app_clip.reshape(B * T, *app_clip.shape[2:])
        mot_flat = mot_clip.reshape(B * T, *mot_clip.shape[2:])
        CHUNK = 64
        if app_flat.shape[0] <= CHUNK:
            feats = self._forward_pair(app_flat, mot_flat)
        else:
            parts = []
            for i in range(0, app_flat.shape[0], CHUNK):
                parts.append(self._forward_pair(app_flat[i:i+CHUNK], mot_flat[i:i+CHUNK]))
            feats = torch.cat(parts, dim=0)
        return self.temporal(feats.reshape(B, T, -1))

# 3. Cross-Attention Fusion v2 
class CrossAttentionFusion_v2(nn.Module):
    def __init__(self, spatial_dim, rppg_dim, proj_dim=FUSION_PROJ_DIM,
                 n_heads=FUSION_HEADS, dropout=DROPOUT):
        super().__init__()
        self.spatial_proj = nn.Sequential(nn.Linear(spatial_dim, proj_dim), nn.LayerNorm(proj_dim))
        self.rppg_proj    = nn.Sequential(nn.Linear(rppg_dim,   proj_dim), nn.LayerNorm(proj_dim))

        self.attn_s2r = nn.MultiheadAttention(proj_dim, n_heads, dropout=dropout, batch_first=True)
        self.attn_r2s = nn.MultiheadAttention(proj_dim, n_heads, dropout=dropout, batch_first=True)

        self.ffn_s = nn.Sequential(nn.Linear(proj_dim, proj_dim), nn.GELU(), nn.Dropout(dropout))
        self.ffn_r = nn.Sequential(nn.Linear(proj_dim, proj_dim), nn.GELU(), nn.Dropout(dropout))
        self.norm_s = nn.LayerNorm(proj_dim)
        self.norm_r = nn.LayerNorm(proj_dim)

        self.reliability_gate = nn.Sequential(
            nn.Linear(proj_dim * 2, proj_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(proj_dim // 2, 1), nn.Sigmoid())

        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim * 2),
            nn.Linear(proj_dim * 2, proj_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(proj_dim // 2, 64), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1))

    def forward(self, spatial_embed, rppg_embed, return_gate=False):
        s = self.spatial_proj(spatial_embed).unsqueeze(1)
        r = self.rppg_proj(rppg_embed).unsqueeze(1)

        s_att, _ = self.attn_s2r(s, r, r)
        r_att, _ = self.attn_r2s(r, s, s)

        s_out = self.norm_s(s.squeeze(1) + self.ffn_s(s_att.squeeze(1)))
        r_out = self.norm_r(r.squeeze(1) + self.ffn_r(r_att.squeeze(1)))

        gate = self.reliability_gate(torch.cat([s_out, r_out], dim=1))
        fused = torch.cat([s_out, gate * r_out], dim=1)

        logits = self.classifier(fused).squeeze(-1)
        if return_gate: return logits, gate.squeeze(-1)
        return logits

# 4. Mô hình Hợp nhất Hoàn chỉnh 
class DeepfakeFusionModel(nn.Module):
    def __init__(self, sp_backbone, rppg_backbone, sp_dim=1024, rppg_dim=256):
        super().__init__()
        self.spatial_backbone = sp_backbone
        self.rppg_backbone    = rppg_backbone
        self.fusion           = CrossAttentionFusion_v2(sp_dim, rppg_dim)
        self.spatial_frozen = True
        self.rppg_frozen    = True

    def set_trainable(self, spatial_tr, rppg_tr):
        for p in self.spatial_backbone.parameters(): p.requires_grad = spatial_tr
        for p in self.rppg_backbone.parameters():    p.requires_grad = rppg_tr
        self.spatial_frozen = not spatial_tr
        self.rppg_frozen    = not rppg_tr

    def train(self, mode=True):
        super().train(mode)
        if self.spatial_frozen: self.spatial_backbone.eval()
        if self.rppg_frozen:    self.rppg_backbone.eval()
        return self

    def encode_spatial(self, frames):
        B, N = frames.shape[:2]
        flat = frames.reshape(B * N, *frames.shape[2:])
        ctx = torch.no_grad() if self.spatial_frozen else torch.enable_grad()
        with ctx:
            feats = self.spatial_backbone(flat)
        return feats.reshape(B, N, -1).mean(1)

    def encode_rppg(self, app, mot):
        ctx = torch.no_grad() if self.rppg_frozen else torch.enable_grad()
        with ctx:
            return self.rppg_backbone.forward_features(app, mot)

    def forward(self, sp_frames, app, mot, return_gate=False):
        s = self.encode_spatial(sp_frames)
        r = self.encode_rppg(app, mot)
        return self.fusion(s, r, return_gate=return_gate)


# 5. Khởi tạo & Nạp Trọng số (Hỗ trợ Resume Stage 1) 
spatial_full = SpatialTransformerDetector(pretrained=False)
spatial_backbone = spatial_full.backbone
SPATIAL_EMBED_DIM = spatial_backbone.num_features

rppg_backbone = DeepPhysCAN_v2()
RPPG_EMBED_DIM = rppg_backbone.embed_dim

fusion_model = DeepfakeFusionModel(spatial_backbone, rppg_backbone, SPATIAL_EMBED_DIM, RPPG_EMBED_DIM).to(DEVICE)

if START_FROM_STAGE >= 2 and has_stage1:
    print(f"\n[RESUME MODE] Nạp toàn bộ trọng số Stage 1 từ: {FUSION_STAGE1_CKPT}")
    ckpt_s1 = torch.load(FUSION_STAGE1_CKPT, map_location="cpu", weights_only=False)
    st = ckpt_s1.get("model_state", ckpt_s1)
    res = fusion_model.load_state_dict(st, strict=False)
    print(f"Kết quả nạp Stage 1: {res}")
    print(f"Đã sẵn sàng chuyển sang Stage {START_FROM_STAGE} để finetune!")
else:
    print(f"\n[SCRATCH/STAGE 1 MODE] Nạp 2 backbone riêng lẻ:")
    ckpt_sp = torch.load(SPATIAL_CKPT_PATH, map_location="cpu", weights_only=False)
    spatial_full.load_state_dict(ckpt_sp.get("model_state", ckpt_sp))
    print("- Spatial backbone loaded (embed_dim={SPATIAL_EMBED_DIM})")

    ckpt_rppg = torch.load(RPPG_CKPT_PATH, map_location="cpu", weights_only=False)
    rppg_backbone.load_state_dict(ckpt_rppg.get("model_state", ckpt_rppg))
    print("- rPPG backbone loaded (embed_dim={RPPG_EMBED_DIM})")

n_all = sum(p.numel() for p in fusion_model.parameters())
print(f"Tổng tham số mô hình: {n_all/1e6:.2f}M")


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, confusion_matrix
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def calc_eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

def find_eer_threshold(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    return float(thresholds[int(np.nanargmin(np.abs(fnr - fpr)))])

def calc_far_frr(y_true, y_score, threshold):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    far = fp / max(fp + tn, 1)
    frr = fn / max(fn + tp, 1)
    return far, frr

def calc_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(int)
    auc = roc_auc_score(y_true, y_score)
    acc = accuracy_score(y_true, y_pred)
    eer = calc_eer(y_true, y_score)
    eer_thr = find_eer_threshold(y_true, y_score)
    far05, frr05 = calc_far_frr(y_true, y_score, 0.5)
    far_eer, frr_eer = calc_far_frr(y_true, y_score, eer_thr)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return {
        "auc": auc, "acc": acc, "eer": eer, "eer_threshold": eer_thr,
        "far_at_0.5": far05, "frr_at_0.5": frr05,
        "far_at_eer": far_eer, "frr_at_eer": frr_eer,
        "cm_at_0.5": cm, "roc_fpr": fpr, "roc_tpr": tpr
    }


In [ ]:
def smooth_labels(labels, eps=LABEL_SMOOTH_EPS):
    return labels * (1 - eps) + 0.5 * eps

def mixup_data(sp, app, mot, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0: return sp, app, mot, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(sp.size(0), device=sp.device)
    return lam*sp + (1-lam)*sp[idx], lam*app + (1-lam)*app[idx], lam*mot + (1-lam)*mot[idx], labels, labels[idx], lam

class ModelEma:
    def __init__(self, model, decay=EMA_DECAY):
        self.ema_model = copy.deepcopy(model).eval().to(DEVICE)
        for p in self.ema_model.parameters(): p.requires_grad_(False)
        self.decay = decay
    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema_model.state_dict().items():
            m = msd[k].detach()
            if v.dtype.is_floating_point: v.mul_(self.decay).add_(m, alpha=1-self.decay)
            else: v.copy_(m)

@torch.no_grad()
def evaluate(eval_model, loader, criterion):
    eval_model.eval()
    y_true, y_score, losses = [], [], []
    for sp, app, mot, label, _ in loader:
        sp, app, mot, label = sp.to(DEVICE), app.to(DEVICE), mot.to(DEVICE), label.to(DEVICE)
        logits = eval_model(sp, app, mot)
        losses.append(criterion(logits, label).item())
        y_score.append(torch.sigmoid(logits).item())
        y_true.append(label.item())
    metrics = calc_metrics(y_true, y_score)
    metrics["loss"] = float(np.mean(losses))
    return metrics

criterion = nn.BCEWithLogitsLoss()
ema = ModelEma(fusion_model, decay=EMA_DECAY)

BEST_PATH = os.path.join(OUT_DIR, "fusion_model_best.pth")
FINAL_PATH = os.path.join(OUT_DIR, "fusion_model_final.pth")

history = {"stage": [], "epoch": [], "train_loss": [],
           "val_loss_raw": [], "val_auc_raw": [], "val_eer_raw": [], "val_acc_raw": [],
           "val_loss_ema": [], "val_auc_ema": [], "val_eer_ema": [], "val_acc_ema": [],
           "used_ema": []}

overall_best_val_auc = -1.0
global_epoch = 0
MIN_EPOCHS = {"stage1_freeze": 4, "stage2_rppg_ft": 4, "stage3_full_ft": 3}

# Định nghĩa các giai đoạn huấn luyện:
# (stage_name, epochs, train_spatial?, train_rppg?, lr_fusion, lr_spatial, lr_rppg)
all_stages = [
    ("stage1_freeze",  EPOCHS_STAGE1, False, False, LR_FUSION,       0.0,                0.0),
    ("stage2_rppg_ft", EPOCHS_STAGE2, False, True,  LR_FUSION * 0.5, 0.0,                LR_FINETUNE_RPPG),
    ("stage3_full_ft", EPOCHS_STAGE3, True,  True,  LR_FINETUNE_ALL_FUS, LR_FINETUNE_ALL_SP, LR_FINETUNE_RPPG * 0.5),
]

# Lọc các stage cần chạy dựa theo START_FROM_STAGE
stages_to_run = all_stages[START_FROM_STAGE - 1:]

print(f"\n{'='*65}")
print(f"BẮT ĐẦU HUẤN LUYỆN GIAI ĐOẠN: {[s[0] for s in stages_to_run]}")
print(f"{'='*65}")

for stage_name, n_epochs, sp_tr, rppg_tr, lr_fus, lr_sp, lr_rppg in stages_to_run:
    fusion_model.set_trainable(sp_tr, rppg_tr)
    n_trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
    print(f"\n{'='*60}")
    print(f"=== BẮT ĐẦU {stage_name} ({n_epochs} epochs) | Trainable params: {n_trainable/1e6:.2f}M ===")
    print(f"{'='*60}")

    # Chuẩn hóa: Reset patience và stage_best_auc RIÊNG CHO MỖI STAGE
    patience_ctr = 0
    stage_best_val_auc = -1.0
    min_ep = MIN_EPOCHS.get(stage_name, 3)

    param_groups = [{"params": fusion_model.fusion.parameters(), "lr": lr_fus}]
    if sp_tr and lr_sp > 0:
        param_groups.append({"params": fusion_model.spatial_backbone.parameters(), "lr": lr_sp})
    if rppg_tr and lr_rppg > 0:
        param_groups.append({"params": fusion_model.rppg_backbone.parameters(), "lr": lr_rppg})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    n_opt_steps = max(len(train_loader) // GRAD_ACCUM, 1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs * n_opt_steps, eta_min=1e-7)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    for epoch in range(1, n_epochs + 1):
        global_epoch += 1
        fusion_model.train()
        running_loss, n_batches = 0.0, 0
        t0 = time.time()
        optimizer.zero_grad()

        for step, (sp, app, mot, label) in enumerate(train_loader):
            sp, app, mot, label = sp.to(DEVICE), app.to(DEVICE), mot.to(DEVICE), label.to(DEVICE)

            # Mixup (bật ở stage 1 & 2, stage 3 tắt để hội tụ dứt khoát)
            if stage_name != "stage3_full_ft" and MIXUP_ALPHA > 0:
                sp, app, mot, y_a, y_b, lam = mixup_data(sp, app, mot, label)
                y_a, y_b = smooth_labels(y_a), smooth_labels(y_b)
            else:
                y_a = smooth_labels(label); y_b = y_a; lam = 1.0

            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                logits = fusion_model(sp, app, mot)
                loss = (lam * criterion(logits, y_a) + (1-lam) * criterion(logits, y_b)) / GRAD_ACCUM

            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(); scheduler.step()
                ema.update(fusion_model)

            running_loss += loss.item() * GRAD_ACCUM
            n_batches += 1

        train_loss = running_loss / max(n_batches, 1)
        val_raw = evaluate(fusion_model, val_loader, criterion)
        val_ema = evaluate(ema.ema_model, val_loader, criterion)
        use_ema = val_ema["auc"] >= val_raw["auc"]
        val = val_ema if use_ema else val_raw

        history["stage"].append(stage_name); history["epoch"].append(global_epoch)
        history["train_loss"].append(train_loss)
        history["val_loss_raw"].append(val_raw["loss"]); history["val_auc_raw"].append(val_raw["auc"])
        history["val_eer_raw"].append(val_raw["eer"]);   history["val_acc_raw"].append(val_raw["acc"])
        history["val_loss_ema"].append(val_ema["loss"]); history["val_auc_ema"].append(val_ema["auc"])
        history["val_eer_ema"].append(val_ema["eer"]);   history["val_acc_ema"].append(val_ema["acc"])
        history["used_ema"].append(use_ema)

        dt = time.time() - t0
        print(f"[{stage_name} {epoch:02d}/{n_epochs}] loss={train_loss:.4f} "
              f"auc_raw={val_raw['auc']:.4f} auc_ema={val_ema['auc']:.4f} "
              f"eer={val['eer']:.4f} | {'EMA' if use_ema else 'raw'} | {dt:.0f}s")

        save_st = ema.ema_model.state_dict() if use_ema else fusion_model.state_dict()
        checkpoint_dict = {
            "model_state": save_st, "epoch": global_epoch, "stage": stage_name,
            "used_ema": use_ema, "val_metrics": val,
            "spatial_embed_dim": SPATIAL_EMBED_DIM, "rppg_embed_dim": RPPG_EMBED_DIM
        }

        # Lưu checkpoint tốt nhất của riêng Stage này
        if val["auc"] > stage_best_val_auc:
            stage_best_val_auc = val["auc"]
            patience_ctr = 0
            stage_ckpt_path = os.path.join(OUT_DIR, f"fusion_model_{stage_name}_best.pth")
            torch.save(checkpoint_dict, stage_ckpt_path)
            print("-> Đã lưu Best {stage_name} (val_auc={stage_best_val_auc:.4f})")
        else:
            patience_ctr += 1

        # Lưu checkpoint tốt nhất toàn cục
        if val["auc"] > overall_best_val_auc:
            overall_best_val_auc = val["auc"]
            torch.save(checkpoint_dict, BEST_PATH)
            print("-> Đã cập nhật OVERALL BEST (val_auc={overall_best_val_auc:.4f})")

        # Kiểm tra Early Stopping trong phạm vi Stage này
        if epoch >= min_ep and patience_ctr >= PATIENCE:
            print(f"Early stopping tại {stage_name} (epoch {epoch}) vì không cải thiện sau {PATIENCE} epochs.")
            break

# Luôn luôn lưu checkpoint cuối cùng
final_st = ema.ema_model.state_dict()
torch.save({
    "model_state": final_st, "epoch": global_epoch, "stage": "final",
    "used_ema": True, "val_metrics": val,
    "spatial_embed_dim": SPATIAL_EMBED_DIM, "rppg_embed_dim": RPPG_EMBED_DIM
}, FINAL_PATH)
print(f"\nĐã lưu checkpoint cuối cùng tại: {FINAL_PATH}")

pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "train_history.csv"), index=False)
print(f"Đã lưu train_history.csv | Overall Best val_auc: {overall_best_val_auc:.4f}")


In [ ]:
@torch.no_grad()
def evaluate_with_tta(model, loader, use_tta=True):
    model.eval()
    y_true, y_score = [], []
    for sp, app, mot, label, _ in tqdm(loader, desc="Test"):
        sp, app, mot = sp.to(DEVICE), app.to(DEVICE), mot.to(DEVICE)
        logits_list = [model(sp, app, mot)]
        if use_tta:
            # TTA 1: Lật ngang
            logits_list.append(model(torch.flip(sp, [4]), torch.flip(app, [4]), torch.flip(mot, [4])))
            # TTA 2: Nửa đầu clip
            T = app.shape[1]; half = max(T // 2, 1)
            logits_list.append(model(sp[:, :sp.shape[1]//2+1], app[:, :half], mot[:, :half]))
        prob = torch.stack([torch.sigmoid(l) for l in logits_list]).mean()
        y_score.append(prob.item()); y_true.append(label.item())
    return np.array(y_true), np.array(y_score)

# Nạp model tốt nhất để test
eval_ckpt_path = BEST_PATH if os.path.exists(BEST_PATH) else FINAL_PATH
print(f"Nạp model đánh giá: {eval_ckpt_path}")
ckpt = torch.load(eval_ckpt_path, map_location=DEVICE, weights_only=False)
fusion_model.load_state_dict(ckpt["model_state"])
print(f"Loaded: epoch={ckpt.get('epoch', '?')} stage={ckpt.get('stage', '?')} (val_auc={ckpt.get('val_metrics', {}).get('auc', 0):.4f})")

y_true, y_score = evaluate_with_tta(fusion_model, test_loader, use_tta=True)
test_m = calc_metrics(y_true, y_score)

print("\n" + "="*50)
print("=== KẾT QUẢ TEST (FF++ C23) — DF-FusionModel CAN v6 + TTA ===")
print("="*50)
print(f"Accuracy : {test_m['acc']*100:.2f}%")
print(f"AUC      : {test_m['auc']*100:.2f}%")
print(f"EER      : {test_m['eer']*100:.2f}%")
print(f"FAR@0.5  : {test_m['far_at_0.5']*100:.2f}%")
print(f"FRR@0.5  : {test_m['frr_at_0.5']*100:.2f}%")

cm = test_m["cm_at_0.5"]
print(f"\nConfusion Matrix: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")

# Breakdown theo phương pháp fake
test_ordered = test_df.reset_index(drop=True)
rows = []
real_mask = test_ordered["label"].values == 0
for method in sorted(test_ordered.loc[~real_mask, "method"].unique()):
    mm = test_ordered["method"].values == method
    sel = real_mask | mm
    m = calc_metrics(y_true[sel], y_score[sel])
    rows.append({"method": method, "auc": m["auc"], "eer": m["eer"]})
bd = pd.DataFrame(rows).sort_values("auc")
bd.to_csv(os.path.join(OUT_DIR, "auc_breakdown_by_method.csv"), index=False)
print("\n=== Breakdown theo phương pháp ===")
print(bd.to_string(index=False))

print("\n=== Bảng so sánh tổng hợp ===")
comp = pd.DataFrame([
    {"model": "Spatial only (Swin)",  "acc": 0.9657, "auc": 0.9926, "eer": 0.0286},
    {"model": "rPPG only (CAN v3)",   "acc": 0.9614, "auc": 0.9912, "eer": 0.0357},
    {"model": "DF-FusionModel CAN v6", "acc": test_m["acc"], "auc": test_m["auc"], "eer": test_m["eer"]},
])
print(comp.to_string(index=False))
comp.to_csv(os.path.join(OUT_DIR, "ablation_table_A.csv"), index=False)

# ---- Confusion Matrix CHUẨN HÓA ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.0, 5.2), dpi=150)
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=10)
classes = ["Real (0)", "Fake (1)"]
ax.set_xticks([0, 1]); ax.set_xticklabels(classes, fontsize=12, fontweight="bold")
ax.set_yticks([0, 1]); ax.set_yticklabels(classes, fontsize=12, fontweight="bold")
total = np.sum(cm); thresh = cm.max() / 2.0
labels_tag = [["TN", "FP"], ["FN", "TP"]]
for i in range(2):
    for j in range(2):
        count = cm[i, j]
        pct = count / total * 100 if total > 0 else 0
        tag = labels_tag[i][j]
        color = "white" if count > thresh else "#08306b"
        ax.text(j, i, f"{tag}\n{count:,}\n({pct:.1f}%)", ha="center", va="center", color=color,
                fontsize=13, fontweight="bold", linespacing=1.3)
ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold", labelpad=10)
ax.set_ylabel("True Label", fontsize=12, fontweight="bold", labelpad=10)
ax.set_title("Confusion Matrix @ threshold=0.5\nDF-FusionModel CAN v6", fontsize=13, fontweight="bold", pad=12)
ax.set_xticks(np.arange(-.5, 2, 1), minor=True)
ax.set_yticks(np.arange(-.5, 2, 1), minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2.5)
ax.tick_params(which="minor", bottom=False, left=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=200, bbox_inches='tight')
plt.show()
plt.close()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 
# 1. Vẽ đồ thị tiến trình huấn luyện (Training Curves)
# 
hist_csv = os.path.join(OUT_DIR, "train_history.csv")
if os.path.exists(hist_csv):
    hist_df = pd.read_csv(hist_csv)
    epochs = hist_df["epoch"].values if "epoch" in hist_df.columns else np.arange(1, len(hist_df) + 1)
    train_loss = hist_df["train_loss"].values if "train_loss" in hist_df.columns else None

    val_loss, vl_label = None, "Val Loss"
    for col in ["val_loss", "val_loss_ema", "val_loss_raw"]:
        if col in hist_df.columns:
            val_loss = hist_df[col].values
            if "ema" in col: vl_label = "Val Loss (EMA)"
            elif "raw" in col: vl_label = "Val Loss (Raw)"
            break

    val_auc = next((hist_df[c].values for c in ["val_auc", "val_auc_ema", "val_auc_raw"] if c in hist_df.columns), None)
    val_eer = next((hist_df[c].values for c in ["val_eer", "val_eer_ema", "val_eer_raw"] if c in hist_df.columns), None)
    val_acc = next((hist_df[c].values for c in ["val_acc", "val_acc_ema", "val_acc_raw"] if c in hist_df.columns), None)

    fig, axes = plt.subplots(1, 4, figsize=(22, 5), dpi=120)

    # Panel 1: Loss
    ax = axes[0]
    if train_loss is not None:
        ax.plot(epochs, train_loss, "o-", color="#1f77b4", lw=2, ms=4, label="Train Loss")
    if val_loss is not None:
        ax.plot(epochs, val_loss, "s--", color="#d62728", lw=2, ms=4, label=vl_label)
        best_i = int(np.argmin(val_loss))
        ax.scatter([epochs[best_i]], [val_loss[best_i]], color="#d62728", s=100, zorder=5,
                   edgecolors="black", marker="*", label=f"Min Val Loss ({val_loss[best_i]:.4f})")
    ax.set_title("Loss (Train vs Val)", fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.grid(True, ls=":", alpha=0.6); ax.legend(fontsize=8, loc="upper right")

    # Panel 2: AUC
    ax = axes[1]
    if val_auc is not None:
        ax.plot(epochs, val_auc, "^-", color="#2ca02c", lw=2, ms=4, label="Val AUC")
        best_i = int(np.argmax(val_auc))
        ax.scatter([epochs[best_i]], [val_auc[best_i]], color="#2ca02c", s=120, zorder=5,
                   edgecolors="black", marker="*", label=f"Best AUC ({val_auc[best_i]*100:.2f}%)")
    ax.set_title("Val AUC", fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("AUC")
    ax.grid(True, ls=":", alpha=0.6); ax.legend(fontsize=8, loc="lower right")

    # Panel 3: EER
    ax = axes[2]
    if val_eer is not None:
        ax.plot(epochs, val_eer * 100, "v-", color="#9467bd", lw=2, ms=4, label="Val EER (%)")
        best_i = int(np.argmin(val_eer))
        ax.scatter([epochs[best_i]], [val_eer[best_i] * 100], color="#9467bd", s=120, zorder=5,
                   edgecolors="black", marker="*", label=f"Min EER ({val_eer[best_i]*100:.2f}%)")
    ax.set_title("Val EER (lower is better)", fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("EER (%)")
    ax.grid(True, ls=":", alpha=0.6); ax.legend(fontsize=8, loc="upper right")

    # Panel 4: Accuracy
    ax = axes[3]
    if val_acc is not None:
        ax.plot(epochs, val_acc * 100, "d-", color="#17becf", lw=2, ms=4, label="Val Acc (%)")
        best_i = int(np.argmax(val_acc))
        ax.scatter([epochs[best_i]], [val_acc[best_i] * 100], color="#17becf", s=120, zorder=5,
                   edgecolors="black", marker="*", label=f"Best Acc ({val_acc[best_i]*100:.2f}%)")
    ax.set_title("Val Accuracy", fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.grid(True, ls=":", alpha=0.6); ax.legend(fontsize=8, loc="lower right")

    plt.suptitle("Training Progress — DF-FusionModel CAN v6", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    curves_path = os.path.join(OUT_DIR, "training_curves.png")
    plt.savefig(curves_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Đã lưu biểu đồ: {curves_path}")

# 
# 2. Đóng gói ZIP toàn bộ Output cho tiện tải về máy
# 
zip_file = "/kaggle/working/fusion_model_output.zip"
print(f"\nĐang đóng gói file ZIP: {zip_file} ...")
os.system(f"cd /kaggle/working && zip -r -q {zip_file} fusion_model/")
if os.path.exists(zip_file):
    size_mb = os.path.getsize(zip_file) / (1024 * 1024)
    print("Đã tạo thành công {zip_file} ({size_mb:.1f} MB)")
    print("Đã hoàn thành tạo tệp lưu trữ kết quả.")
else:
    print("Hoàn tất huấn luyện!")
